<a href="https://colab.research.google.com/github/tri25msb43048-eng/rag-thu-tuc-hanh-chinh1/blob/main/rag_thu_tuc_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG tra cứu thủ tục hành chính (Llamafile + Qdrant)
Chuyển thể từ lab *Introduction to RAG* (Duke University, Alfredo Deza) với dữ liệu thủ tục hành chính cấp xã.

**Các điểm khác so với notebook gốc của lab:**
- Model embedding đa ngữ (`paraphrase-multilingual-MiniLM-L12-v2`) thay cho model tiếng Anh `all-MiniLM-L6-v2`.
- Dùng `query_points` thay cho `search` (đã bị bỏ trong `qdrant-client` mới).
- Tìm kiếm lai: khớp từ khóa cách nói của người dân + độ tương đồng cosine.
- Tài liệu đưa vào tin nhắn `user` có nhãn rõ ràng, thay vì vai `assistant`.
- Bước kiểm tra liên quan trước khi trả lời; không có thủ tục phù hợp thì chuyển Bộ phận Một cửa.
- LLM Qwen2.5-3B-Instruct chạy bằng Llamafile trên CPU.

*Dữ liệu `thu_tuc_mau.csv` là dữ liệu minh họa, chưa đối chiếu văn bản chính thức.*

Chạy lần lượt từng ô, hoặc **Thời gian chạy → Chạy tất cả**.

## 1. Cài thư viện

In [ ]:
!pip install -q qdrant-client sentence-transformers openai pandas

## 2. Tải dữ liệu
Tải `thu_tuc_mau.csv` lên (thư mục bên trái → Upload), hoặc chạy ô dưới để chọn tệp.

In [ ]:
import os
if not os.path.exists('thu_tuc_mau.csv'):
    from google.colab import files
    files.upload()

## 3. Chạy LLM bằng Llamafile (cổng 8081)
- Tải tệp chạy Llamafile mới nhất từ GitHub và model Qwen2.5-3B-Instruct (~2,1 GB). Lần đầu mất vài phút.
- Cổng 8080 trên Colab đã bị dịch vụ nội bộ chiếm, nên dùng 8081.
- Colab không có Vulkan, nên ép chạy CPU (`--gpu disable`).

In [ ]:
import subprocess, time, os, json, urllib.request
from openai import OpenAI

if not os.path.exists('llamafile_bin') or os.path.getsize('llamafile_bin') < 1_000_000:
    rel = json.load(urllib.request.urlopen(
        'https://api.github.com/repos/mozilla-ai/llamafile/releases/latest'))
    url = [a['browser_download_url'] for a in rel['assets']
           if a['name'] == f"llamafile-{rel['tag_name']}"][0]
    print('Đang tải Llamafile', rel['tag_name'])
    subprocess.run(['wget', '-q', '-O', 'llamafile_bin', url], check=True)
os.chmod('llamafile_bin', 0o755)

GGUF = 'https://huggingface.co/Qwen/Qwen2.5-3B-Instruct-GGUF/resolve/main/qwen2.5-3b-instruct-q4_k_m.gguf'
if not os.path.exists('qwen3b.gguf') or os.path.getsize('qwen3b.gguf') < 2_000_000_000:
    print('Đang tải model 3B (~2,1 GB)...')
    subprocess.run(['wget', '-q', '-O', 'qwen3b.gguf', GGUF], check=True)

subprocess.run('pkill -f llamafile', shell=True); time.sleep(2)
log = open('llamafile.log', 'w')
proc = subprocess.Popen(
    ['sh', './llamafile_bin', '-m', 'qwen3b.gguf', '-t', str(os.cpu_count()),
     '--server', '--gpu', 'disable', '--host', '127.0.0.1', '--port', '8081'],
    stdout=log, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
    start_new_session=True)

client = OpenAI(base_url='http://127.0.0.1:8081/v1', api_key='sk-no-key-required')
for i in range(60):  # tối đa 5 phút
    if proc.poll() is not None:
        print(f'Llamafile đã tắt (mã thoát {proc.returncode}). Log:\n')
        print(open('llamafile.log').read()[-3000:]); break
    try:
        r = client.chat.completions.create(model='LLaMA_CPP', max_tokens=20,
                messages=[{'role': 'user', 'content': 'Xin chào'}])
        print('LLM sẵn sàng. Trả lời thử:', r.choices[0].message.content); break
    except Exception:
        print(f'Đang nạp model... {5*(i+1)} giây'); time.sleep(5)

## 4. Đọc dữ liệu và tạo kho vector

In [ ]:
import pandas as pd
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer

TEN_COLLECTION = 'thu_tuc_hanh_chinh'
NGUONG_DIEM = 0.35

# Cách nói của người dân cho từng thủ tục (cán bộ Một cửa bổ sung dần)
TU_KHOA = {
    'VD-01': ['làm giấy khai sinh', 'khai sinh cho con', 'đăng ký cho em bé'],
    'VD-05': ['sao y', 'photo công chứng', 'chứng thực', 'công chứng'],
    'VD-06': ['xác nhận độc thân', 'xác nhận chưa kết hôn'],
}

df = pd.read_csv('thu_tuc_mau.csv').fillna('')
thu_tuc = df.to_dict('records')
THEO_MA = {t['ma']: t for t in thu_tuc}

def van_ban_de_tim(tt):
    return f"{tt['ten']}. {tt['mo_ta']}. {'; '.join(TU_KHOA.get(tt['ma'], []))}. Lĩnh vực {tt['linh_vuc']}."

encoder = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
qdrant = QdrantClient(':memory:')
if qdrant.collection_exists(TEN_COLLECTION):
    qdrant.delete_collection(TEN_COLLECTION)
qdrant.create_collection(
    collection_name=TEN_COLLECTION,
    vectors_config=models.VectorParams(size=encoder.get_sentence_embedding_dimension(),
                                       distance=models.Distance.COSINE),
)
vectors = encoder.encode([van_ban_de_tim(tt) for tt in thu_tuc])
qdrant.upload_points(
    collection_name=TEN_COLLECTION,
    points=[models.PointStruct(id=i, vector=v.tolist(), payload=tt)
            for i, (v, tt) in enumerate(zip(vectors, thu_tuc))],
)
print('Đã nạp', len(thu_tuc), 'thủ tục')

## 5. Tìm kiếm lai
Khớp từ khóa trước (điểm 1,0), rồi bổ sung kết quả theo độ tương đồng cosine. Câu hỏi ngắn như "sao y căn cước" xếp đúng VD-05 nhưng điểm cosine chỉ 0,17, dưới ngưỡng, nên cần khớp từ khóa để không bỏ sót.

In [ ]:
def tim_thu_tuc(cau_hoi, k=3):
    q = cau_hoi.lower()
    khop = [(THEO_MA[ma], 1.0) for ma, ds in TU_KHOA.items()
            if ma in THEO_MA and any(tk in q for tk in ds)]
    da_co = {t['ma'] for t, _ in khop}
    pts = qdrant.query_points(collection_name=TEN_COLLECTION,
                              query=encoder.encode(cau_hoi).tolist(), limit=k).points
    theo_nghia = [(p.payload, p.score) for p in pts
                  if p.score >= NGUONG_DIEM and p.payload['ma'] not in da_co]
    return (khop + theo_nghia)[:k]

for q in ['Làm giấy khai sinh cho con cần mang gì?', 'sao y căn cước ở đâu']:
    print(q)
    for tt, diem in tim_thu_tuc(q):
        print('  ', tt['ma'], tt['ten'], round(diem, 2))

## 6. Kiểm tra liên quan và gọi LLM
1. Hỏi LLM từng thủ tục tìm được có đúng việc người dân muốn làm không (CO/KHONG).
2. Không còn thủ tục nào → chuyển Bộ phận Một cửa, không gọi bước viết câu trả lời.
3. Có thủ tục → LLM viết câu trả lời chỉ từ tài liệu; code tự gắn dòng Nguồn.

In [ ]:
CHI_DAN = ('Bạn là trợ lý tra cứu thủ tục hành chính của UBND xã. '
           'Chỉ dùng thông tin trong phần TÀI LIỆU; không thêm, không suy đoán chi tiết nào khác. '
           'Nếu tài liệu có thủ tục đúng với câu hỏi, trả lời đủ 4 mục: Hồ sơ, Thời hạn, Lệ phí, Nơi nộp. '
           'Nếu tài liệu không có thủ tục phù hợp, nói rõ là không có thông tin và đề nghị '
           'người dân liên hệ Bộ phận Một cửa. Chỉ viết bằng tiếng Việt.')

def dinh_dang(ds):
    return '\n\n'.join(
        f"[{t['ma']}] {t['ten']} (cấp {t['cap']})\n- Hồ sơ: {t['ho_so']}\n"
        f"- Thời hạn: {t['thoi_han']}\n- Lệ phí: {t['le_phi']}\n- Nơi nộp: {t['noi_nop']}"
        for t, _ in ds)

def kiem_tra_lien_quan(cau_hoi, tt):
    r = client.chat.completions.create(
        model='LLaMA_CPP', temperature=0, max_tokens=5,
        messages=[{'role': 'user', 'content':
            f'Người dân hỏi: "{cau_hoi}"\n'
            f"Thủ tục: {tt['ten']} — {tt['mo_ta']}\n"
            'Thủ tục này có đúng là việc người dân muốn làm không? '
            'Chỉ trả lời một từ: CO hoặc KHONG.'}])
    return r.choices[0].message.content.strip().upper().startswith('CO')

def tra_loi(cau_hoi):
    ds = [x for x in tim_thu_tuc(cau_hoi) if kiem_tra_lien_quan(cau_hoi, x[0])]
    if not ds:
        return 'Chưa tìm thấy thủ tục phù hợp. Vui lòng liên hệ Bộ phận Một cửa UBND xã.'
    r = client.chat.completions.create(
        model='LLaMA_CPP', temperature=0.2,
        messages=[{'role': 'system', 'content': CHI_DAN},
                  {'role': 'user', 'content': f'TÀI LIỆU:\n{dinh_dang(ds)}\n\nCÂU HỎI CỦA NGƯỜI DÂN: {cau_hoi}'}])
    nguon = '; '.join(f"[{t['ma']}] {t['ten']}" for t, _ in ds)
    return r.choices[0].message.content + f'\n\nNguồn: {nguon}'

## 7. Chạy thử
Câu 4 nằm ngoài phạm vi kho; kết quả đúng là chuyển Bộ phận Một cửa.

In [ ]:
for q in ['Làm giấy khai sinh cho con cần mang gì?',
          'Tôi muốn xin giấy xác nhận độc thân',
          'sao y căn cước ở đâu',
          'xin cấp giấy phép xây dựng nhà ở']:
    print('Người dân:', q)
    print('Trợ lý:', tra_loi(q), '\n')

## 8. So sánh: điền thông tin từ kho thay vì để LLM viết
Ở mục 7, LLM viết lại hồ sơ, thời hạn, lệ phí từ tài liệu. Khi chạy thử, với câu "sao y căn cước ở đâu", LLM đã sửa *"Bản chính giấy tờ; Bản sao cần chứng thực"* thành *"Căn cước công dân"*, và *"Theo số trang chứng thực"* thành *"Theo số trang căn cước"*.

Cách dưới đây vẫn dùng LLM để kiểm tra liên quan, nhưng code điền thẳng các mục từ kho, nên nội dung đúng từng chữ. Khi còn hơn một thủ tục, chatbot đề nghị người dân xác nhận thay vì tự chọn.

In [ ]:
def tra_loi_tu_kho(cau_hoi):
    ds = [x for x in tim_thu_tuc(cau_hoi) if kiem_tra_lien_quan(cau_hoi, x[0])]
    if not ds:
        return 'Chưa tìm thấy thủ tục phù hợp. Vui lòng liên hệ Bộ phận Một cửa UBND xã.'
    khoi = []
    for t, _ in ds:
        khoi.append(
            f"Thủ tục: {t['ten']} [{t['ma']}]\n"
            f"- Hồ sơ: {t['ho_so']}\n"
            f"- Thời hạn: {t['thoi_han']}\n"
            f"- Lệ phí: {t['le_phi']}\n"
            f"- Nơi nộp: {t['noi_nop']}")
    mo_dau = ('Anh/chị có thể đang hỏi về thủ tục sau:' if len(ds) == 1
              else 'Có thể anh/chị đang hỏi về một trong các thủ tục sau, vui lòng xác nhận:')
    return mo_dau + '\n\n' + '\n\n'.join(khoi)

for q in ['Làm giấy khai sinh cho con cần mang gì?',
          'Tôi muốn xin giấy xác nhận độc thân',
          'sao y căn cước ở đâu',
          'xin cấp giấy phép xây dựng nhà ở']:
    print('Người dân:', q)
    print('Trợ lý:', tra_loi_tu_kho(q), '\n')